# ERA5 to PRISM Downscaling (Georgia)

Baseline climate downscaling framework with temporal ERA5 input and PRISM targets.


## 1. Goal
Estimate high-resolution PRISM temperature fields from coarse ERA5 observations for matched dates over Georgia.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ERA5_PATH = ROOT / 'data_raw' / 'era5_georgia_temp.nc'
PRISM_PATH = ROOT / 'data_raw' / 'prism'
CHECKPOINT_PATH = ROOT / 'checkpoints' / 'cnn_downscaler_best.pt'
RESULTS_DIR = ROOT / 'results' / 'evaluation'
HISTORY_LENGTH = 3

print('ERA5_PATH:', ERA5_PATH)
print('PRISM_PATH:', PRISM_PATH)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('RESULTS_DIR:', RESULTS_DIR)
print('HISTORY_LENGTH:', HISTORY_LENGTH)


## 2. Data (ERA5 vs PRISM)
ERA5 provides coarse global reanalysis fields. PRISM provides higher-resolution regional temperature grids used as supervision targets.


In [ ]:
availability = {
    'ERA5': ERA5_PATH.exists(),
    'PRISM': PRISM_PATH.exists(),
    'Checkpoint': CHECKPOINT_PATH.exists(),
}
for name, exists in availability.items():
    print(f'{name}: {exists}')

if not availability['ERA5'] or not availability['PRISM']:
    display(Markdown('Data files are not available in this environment.'))


## 3. Model overview
The baseline model is a CNN that treats temporal ERA5 history as input channels and predicts a PRISM-resolution map.


In [ ]:
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
    model_config = checkpoint.get('model_config', {}) if isinstance(checkpoint, dict) else {}
    print('Checkpoint model_config:', model_config)
else:
    display(Markdown('Checkpoint is not available in this environment.'))


## 4. Inference pipeline
Load aligned ERA5/PRISM samples, run forward inference, and compute per-sample error metrics.


In [ ]:
if availability['ERA5'] and availability['PRISM'] and availability['Checkpoint']:
    from datasets.prism_dataset import ERA5_PRISM_Dataset
    from models.cnn_downscaler import CNNDownscaler

    dataset = ERA5_PRISM_Dataset(
        era5_path=str(ERA5_PATH),
        prism_path=str(PRISM_PATH),
        history_length=HISTORY_LENGTH,
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    cfg = checkpoint.get('model_config', {}) if isinstance(checkpoint, dict) else {}

    model = CNNDownscaler(
        in_channels=int(cfg.get('in_channels', HISTORY_LENGTH)),
        out_channels=int(cfg.get('out_channels', 1)),
        base_channels=int(cfg.get('base_channels', 32)),
    )
    model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
    model = model.to(device).eval()

    date = dataset.metadata(0).date
    x, y = dataset[0]
    x_batch = x.unsqueeze(0).to(device)
    y_batch = y.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x_batch, target_size=(y.shape[-2], y.shape[-1]))

    rmse = torch.sqrt(torch.mean((pred - y_batch) ** 2)).item()
    mae = torch.mean(torch.abs(pred - y_batch)).item()
    print('Date:', date.date())
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE : {mae:.4f}')
else:
    display(Markdown('Inference step skipped because required files are unavailable.'))


## 5. Results visualization
Load evaluation artifacts from `results/evaluation/` and display image + metrics.


In [ ]:
plot_files = sorted(RESULTS_DIR.glob('comparison_*.png'))
metrics_path = RESULTS_DIR / 'metrics.json'

if plot_files and metrics_path.exists():
    display(Image(filename=str(plot_files[0])))
    metrics = json.loads(metrics_path.read_text())
    display(Markdown(f"RMSE: {metrics['rmse']:.4f} | MAE: {metrics['mae']:.4f} | history_length: {metrics.get('history_length', 'n/a')}"))
else:
    display(Markdown('Run evaluation script to generate results'))
